In [1]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)

print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_new.csv"

# Path to zarr
zarr_dir = r"Q:\NOTEBOOKS\Christine\all_slides\zarr"

Index(['filename', 'rekvnr', 'modtdato', 'rekvprio', 'team', 'sex', 'alder',
       'alder gruppe', 'rekvdato', 'matantal', 'mattype tekst', 'glasalm',
       'immunalm', 'ishutaelalm', 'antglas', 'specfarv', 'stain',
       'snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text',
       'undersoeger_anonymous', 'T_category', 'M_category'],
      dtype='object')


In [2]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

C:\ProgramData\miniconda3\envs\lazyslide_010_py310_uv_2\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
C:\ProgramData\miniconda3\envs\lazyslide_010_py310_uv_2\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
from helper_functions import with_tissue_artifact

# Path to cache file
cache_file = "cache_all_slides.pkl"

df_tissue = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="complete", version="default")
with_tissue = list(set(df_tissue["filename"].tolist()))
print("WSIs with completed default tissue detection: ", len(with_tissue))

df_sub = df_all[df_all["filename"].isin(with_tissue)].copy()

Dataframe subset length:  71416 (tissue, default, complete)
WSIs with completed default tissue detection:  71416


In [5]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_sub, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = df_HE[df_HE['T_category'].apply(len) == 1]

mask_insufficient = (
    (df_HE['M_category'] == "Morphology Not Applicable / Insufficient Tissue") &
    (df_HE['M_category'].apply(lambda x: len(x) == 1 if isinstance(x, (list, str)) else False))
)

df_insufficient = df_HE[mask_insufficient]
df_no_insufficient = df_HE[~mask_insufficient]

In [6]:
from helper_functions import lists2tuples

df_no_insufficient = df_no_insufficient.apply(lists2tuples)
print(len(df_no_insufficient))

24299


In [7]:
df_features = pd.read_csv(output_path)

# Files with error during feature extraction
feature_errors = df_features[df_features['status'] != "feature extraction complete"].copy()
feature_errors = set(feature_errors["wsi_path"])

# Files with successful feature extraction
feature_result = df_features[df_features['status'] == "feature extraction complete"].copy()
feature_result = set(feature_result["wsi_path"])

errors_only = feature_errors - feature_result

# Exclude files, that never had a successfull feature extraction run
df_no_errors = df_no_insufficient[~df_no_insufficient["filename"].isin(errors_only)]
print(len(df_no_errors))

22821


In [8]:
def generate_subset(df, n_samples, n_tissues, feature_summary, model):
    # Get top tissues
    top_n = df['T_category'].value_counts().nlargest(n_tissues).index
    df_top = df[df['T_category'].isin(top_n)]
    
    # Load and filter feature summary
    df_features = pd.read_csv(feature_summary)
    df_features = df_features[df_features['status'] == "feature extraction complete"].copy()
    df_features = df_features[df_features['model'] == model].copy()

    # Set of WSIs with features
    feature_wsi_set = set(df_features["wsi_path"])

    subsets = []

    # Loop through each tissue category
    for tissue, df_category in df_top.groupby('T_category'):

        # 1. Split into with/without features
        df_with_features = df_category[df_category["filename"].isin(feature_wsi_set)]
        df_no_features = df_category[~df_category["filename"].isin(feature_wsi_set)]
        print(f'Samples from {tissue} with {model}-features:', len(df_with_features))
        
        # 2. Sample from with features first
        n_samples_with_features = min(len(df_with_features), n_samples)
        sample_features = df_with_features.sample(n=n_samples_with_features, random_state=42)
    
        # 3. Fallback sampling
        samples_needed = n_samples - n_samples_with_features
        if samples_needed > 0:
            sample_fallback = df_no_features.sample(n=min(len(df_no_features), samples_needed), random_state=42)
        else:
            sample_fallback = pd.DataFrame(columns=df.columns)
    
        # 4. Combine
        combined = pd.concat([sample_features, sample_fallback])

        # Logging
        print(f"{tissue} ({len(df_category)} total):")
        print(f"  {len(sample_features)} with features")
        print(f"  {len(sample_fallback)} fallback")
        print(f"  → Combined: {len(combined)}\n")

        subsets.append(combined)

    # Final combined subset
    df_subset = pd.concat(subsets).reset_index(drop=True)
    print(f"\nFinal combined subset size: {len(df_subset)}")

    return df_subset

In [8]:
result = generate_subset(df_no_errors, n_samples = 100, n_tissues = 10, feature_summary = output_path, model = "h-optimus-0")

# Save to csv
output_file = r"D:\DATA\abmil_exp3.csv"

result.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")

Samples from ('Accessory Digestive Organs',) with h-optimus-0-features: 100
('Accessory Digestive Organs',) (291 total):
  100 with features
  0 fallback
  → Combined: 100

Samples from ('Blood, Bone Marrow and Lymphatic System',) with h-optimus-0-features: 596
('Blood, Bone Marrow and Lymphatic System',) (1883 total):
  100 with features
  0 fallback
  → Combined: 100

Samples from ('Endocrine System',) with h-optimus-0-features: 100
('Endocrine System',) (337 total):
  100 with features
  0 fallback
  → Combined: 100

Samples from ('Female Reproductive System (Maternal)',) with h-optimus-0-features: 100
('Female Reproductive System (Maternal)',) (16408 total):
  100 with features
  0 fallback
  → Combined: 100

Samples from ('Lower Digestive Tract',) with h-optimus-0-features: 100
('Lower Digestive Tract',) (122 total):
  100 with features
  0 fallback
  → Combined: 100

Samples from ('Male Reproductive System',) with h-optimus-0-features: 100
('Male Reproductive System',) (731 total

C:\Users\x9cccj\AppData\Local\Temp\11\ipykernel_13404\4033853863.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([sample_features, sample_fallback])
C:\Users\x9cccj\AppData\Local\Temp\11\ipykernel_13404\4033853863.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([sample_features, sample_fallback])
C:\Users\x9cccj\AppData\Local\Temp\11\ipykernel_13404\4033853863.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA en

In [24]:
# Additional skeletal slides, where len(T_category) more than 1 is allowed

df_skeletal = subset_df(df_sub, "stain", "HE")
df_skeletal = subset_df(df_skeletal, "mattype tekst", "Hist. store")
df_skeletal = subset_df_list(df_skeletal, "T_category", "Skeletal System")
mask = (
    (df_skeletal['M_category'] == "Morphology Not Applicable / Insufficient Tissue") &
    (df_skeletal['T_category'].apply(lambda x: len(x) == 1 if isinstance(x, (list, str)) else False))
)
df_skeletal = df_skeletal[~mask]
df_features = pd.read_csv(output_path)
df_features = df_features[df_features['status'] == "feature extraction complete"].copy()
df_features = df_features[df_features['model'] == "h-optimus-0"].copy()
feature_wsi_set = set(df_features["wsi_path"])
df_skeletal = df_skeletal[df_skeletal["filename"].isin(feature_wsi_set)]
print(len(df_skeletal))

100


In [29]:
df_missing = df_skeletal[~df_skeletal["filename"].isin(result["filename"])]
df_combined = pd.concat([result, df_missing], ignore_index=True)
print(len(df_combined))

# Save to csv
output_file = r"D:\DATA\abmil_exp3.csv"

df_combined.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")

1000
Saved DataFrame to D:\DATA\abmil_exp3.csv


In [9]:
all_filenames = result["filename"].tolist()
print("Number of files: ", len(all_filenames))

Number of files:  991
